# Remarks & Special Thanks

The knowledge portrayed is based on the remarkable book "Large Language Models selbst programmieren", written by Sebastian Raschka. 
I declare: My interest is solely in learning. I am not using ideas and concepts laid out in this book for proprietary economical interests.

Source:
Raschka, Sebastian (2025). Large Language Models selbs programmieren: Mit Python und PyTorch ein eigenes LLM entwickeln. dpunkt.verlag (Rheinwerk Verlag): Bonn.

# Imports

In [152]:
import torch
import re
import pandas as pd
import tiktoken
from torch.utils.data import Dataset, DataLoader

# Hardware used for compute

In [20]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Current available device: {device}")

Current available device: mps


# (1) Working with Text Data

## Import text

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

In [4]:
print("Total number of chars:", len(raw_text))
print(raw_text[:100])

Total number of chars: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


## Tokenizing text

### testing tokenizer

In [5]:
# input text
text = "Hello, world. Is this-a test?"

# tokenizing schema
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)

# tokenized text
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this-a', 'test', '?']


### Preprocessing (applying tokenizer on imported text)

In [10]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(f"Token length of imported text: {len(preprocessed)}")
n = 30
print(f"\nFirst {n} tokens: {preprocessed[:n]}")

Token length of imported text: 4690

First 30 tokens: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Converting tokens into Token-IDs

In [73]:
all_tokens = sorted(set(preprocessed))
vocab_size = len(all_tokens)
print(f"Size of words: {vocab_size}")
print(f"\nFirst {n} alphabetically ordered tokens {all_tokens[:n]}")

Size of words: 1130

First 30 alphabetically ordered tokens ['!', '"', "'", '(', ')', ',', '--', '.', ':', ';', '?', 'A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be', 'Begin', 'Burlington', 'But', 'By', 'Carlo', 'Chicago', 'Claude', 'Come', 'Croft', 'Destroyed']


In [74]:
vocab = {token: ID for ID, token in enumerate(all_tokens)}

In [75]:
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: ID for ID, token in enumerate(all_tokens)}

In [76]:
df = pd.DataFrame(vocab.items(), columns=["Token", "Token ID"])
df

,Token,Token ID
0,!,0
1,"""",1
2,',2
3,(,3
4,),4
...,...,...
1127,younger,1127
1128,your,1128
1129,yourself,1129
1130,<|endoftext|>,1130


## How to build a tokenizer (V2)

In [77]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        # saves vocabulary as class attribute to access during Encode-Decode-methods
        self.str_to_int = vocab
        # creates inverse vocabulary (Token IDs to original text token)
        self.int_to_str = {integer:string for string,integer in vocab.items()}

    # transforms input text to Token IDs
    def encode(self, text):
        preprocessed = re.split(r'([,.;:?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        # unknown words are replaced by <|unk|> token
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        
        ID_list = [self.str_to_int[string] for string in preprocessed]
        return ID_list

    # converts Token IDs back to text
    def decode(self, ID_list):
        text = " ".join([self.int_to_str[integer] for integer in ID_list])
        # removes space before defined punctuation marks
        text = re.sub(r'\s+([,.;:?!"()\'])', r'\1', text)
        return text

### Testing tokenizer (V2)


In [79]:
tokenizer = SimpleTokenizerV2(vocab)

text = """
"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride. 
"The last but one," she corrected herself--"but the other doesn't count, because he destroyed it."
"""

ID_list = tokenizer.encode(text)
print(f"Text to Token IDs:\n{ID_list}")
print(f"\n\nToken IDs to text:\n{tokenizer.decode(ID_list)}")

Text to Token IDs:
[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7, 1, 93, 602, 239, 729, 5, 1, 876, 291, 542, 6, 1, 239, 988, 735, 356, 2, 970, 294, 5, 205, 533, 330, 585, 7, 1]


Token IDs to text:
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride." The last but one," she corrected herself --" but the other doesn' t count, because he destroyed it."


### Testing <|endoftext|> token

In [82]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

tokenizer = SimpleTokenizerV2(vocab)
ID_list = tokenizer.encode(text)
print(f"Text to Token IDs:\n{ID_list}")

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
Text to Token IDs:
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [87]:
from importlib.metadata import version
print(f"tiktoken version: {version('tiktoken')}")

tiktoken version: 0.12.0


### Using `tiktoken`

In [126]:
tokenizer = tiktoken.get_encoding("gpt2")
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."

# encoding
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

# decoding
strings = tokenizer.decode(integers)
print(strings)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [127]:
tokens = [tokenizer.decode([i]) for i in integers]

df = pd.DataFrame({"Token": tokens, "Token ID": integers})
df

,Token,Token ID
0,Hello,15496
1,",",11
2,do,466
3,you,345
4,like,588
5,tea,8887
6,?,30
7,,220
8,<|endoftext|>,50256
9,In,554


## Data sampling

In [129]:
with open("the-verdict.txt", "r",encoding="utf-8") as file:
    raw_text = file.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [137]:
enc_sample = enc_text[50:]

# context size determines how many tokens as part of input are taken into account
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x:{x}")
print(f"y:     {y}")

x:[290, 4920, 2241, 287]
y:     [4920, 2241, 287, 257]


In [150]:
# prediction of next word
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

print("\n")    

# conversion of Token IDs into text
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [143]:
enc_sample[:10]

[290, 4920, 2241, 287, 257, 4489, 64, 319, 262, 34686]

### Dataset for stacked inputs and targets

In [162]:
class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_IDs = []
        self.target_IDs = []

        # tokenizing of whole text
        token_IDs = tokenizer.encode(text)

        # using a string window to cut text into overlapping sequences of `max_length`
        for i in range(0, len(token_IDs) - max_length, stride):
            input_chunk = token_IDs[i:i + max_length]
            target_chunk = token_IDs[i + 1:i + max_length + 1]
            self.input_IDs.append(torch.tensor(input_chunk))
            self.target_IDs.append(torch.tensor(target_chunk))

    # total of lines in dataset
    def __len__(self):
        return len(self.input_IDs)

    # single line in dataset
    def __getitem__(self, index):
        return self.input_IDs[index], self.target_IDs[index]

In [163]:
def create_dataloader_v1(
    text, batch_size=4, max_length=256, stride=128,
    shuffle=True, drop_last=True, num_workers=0
):
    # init of tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")
    # creates dataset
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last, # last batch dropped, if short than defined `batch_size` (reduces loss peaks during training)
        num_workers=num_workers # num of CPU-processes used during preprocessing
    )
    return dataloader
    
    

In [166]:
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
# converts dataloader into Python iterator to call next entry with `next()`
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [167]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


### Sample of `batch_size` > 1

#### `max_length=4` and `stride=1`: high token overlap

In [169]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"Inputs:\n\t{inputs}")
print(f"Targets:\n\t{targets}")

Inputs:
	tensor([[   40,   367,  2885,  1464],
        [  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257]])
Targets:
	tensor([[  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257],
        [10899,  2138,   257,  7026]])


#### `max_length=4` and `stride=4`: no token overlap

with value of `stride` corresponding to value of `max_length` overlaps between batches are avoided (overlaps increase overfitting)

In [171]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"Inputs:\n\t{inputs}")
print(f"Targets:\n\t{targets}")

Inputs:
	tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
	tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Creating Token-Embeddings

embedding weights are randomly initialized (starting point for LLM learning process)

### Simple example of randomly initialized Embedding Weights of dimension 3

In [174]:
input_IDs = torch.tensor([2,3,5,1])

vocab_size = 6
output_dim = 3

torch.manual_seed(42)
embedding_layer = torch.nn.Embedding(vocab_size,output_dim)
# weight matrix of embedded layer (size of 6, due to `vocab_size`, with a dimension of 3 for each)
print(embedding_layer.weight)

# weight matrix applied on one Token ID, i.e. ID 3
print(embedding_layer(torch.tensor([3])))

# weight matrix applied on input IDs 
print(embedding_layer(input_IDs))

Parameter containing:
tensor([[ 1.9269,  1.4873, -0.4974],
        [ 0.4396, -0.7581,  1.0783],
        [ 0.8008,  1.6806,  0.3559],
        [-0.6866,  0.6105,  1.3347],
        [-0.2316,  0.0418, -0.2516],
        [ 0.8599, -0.3097, -0.3957]], requires_grad=True)
tensor([[-0.6866,  0.6105,  1.3347]], grad_fn=<EmbeddingBackward0>)
tensor([[ 0.8008,  1.6806,  0.3559],
        [-0.6866,  0.6105,  1.3347],
        [ 0.8599, -0.3097, -0.3957],
        [ 0.4396, -0.7581,  1.0783]], grad_fn=<EmbeddingBackward0>)


### 256-dim example

In [179]:
vocab_size = 50257
output_dim = 256
batch_size = 8
max_length = 4
shuffle = False

token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)
dataloader = create_dataloader_v1(raw_text, batch_size=batch_size, max_length=max_length, shuffle=shuffle)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print(f"Token IDs:\n\t{inputs}")
print(f"\nInputs shape:\n\t{inputs.shape}")

token_embeddings = token_embedding_layer(inputs)
print(f"\nToken embeddings shape:\n\t{token_embeddings.shape}")

Token IDs:
	tensor([[   40,   367,  2885,  1464],
        [  286,   616,  4286,   705],
        [10197,   832,   262, 46475],
        [ 4150,     8,  3688,   284],
        [  271, 10899,   550,   366],
        [ 1021,   757,   438, 10919],
        [  314,  4752,   340,  6777],
        [  423,  4750,   326,  9074]])

Inputs shape:
	torch.Size([8, 4])

Token embeddings shape:
	torch.Size([8, 4, 256])


### Absolute embedding-approach of GPT models


In [181]:
# `context_length` equals input size supported by LLM 
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
# placeholder vector (sequence of nums up to maximum imput length (-1))
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

# note: in practice, input text can be larger than supported context length

torch.Size([4, 256])


In [188]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)
print(f"Token embeddings:\n\t{token_embeddings}")
print(f"\nPositional embeddings:\n\t{pos_embeddings}")
print(f"\nInput embeddings (`token_embeddings + pos_embeddings`):\n\t{input_embeddings}")

torch.Size([8, 4, 256])
Token embeddings:
	tensor([[[ 1.0660,  0.2672, -0.5655,  ...,  0.9059, -0.1661, -0.6976],
         [-0.8826, -1.2952,  1.3008,  ..., -1.8407, -0.1071, -0.9257],
         [ 1.0141, -0.9804, -0.8453,  ...,  1.3521,  0.5144, -1.0465],
         [-0.5057,  0.1638,  1.3839,  ..., -0.5419,  1.0535,  0.9612]],

        [[-0.6353, -1.0762,  0.2975,  ...,  0.9649, -0.8755,  0.3865],
         [ 1.2939, -1.5593,  0.1875,  ...,  0.6222,  0.6592, -0.6366],
         [-0.4417, -0.8772,  0.1559,  ..., -0.3718, -0.8103,  0.1368],
         [-1.2676,  0.5410,  0.6506,  ...,  0.7905, -0.3457,  1.1010]],

        [[-1.0726, -1.7081,  1.1890,  ..., -0.6183, -0.6063, -1.5884],
         [-0.1158,  1.3644,  0.3069,  ...,  0.8668, -0.3771,  0.6315],
         [ 1.0351, -1.0199,  0.4319,  ...,  0.0825, -0.1981,  1.1637],
         [ 0.3759,  0.5865, -0.8039,  ...,  1.9121, -0.2826, -0.4555]],

        ...,

        [[ 2.4448, -0.7193, -0.8914,  ...,  0.7403, -1.2147,  1.7504],
         [ 2.7

# (2) Attention Mechanism